# Разведочный анализ датасета HateXplain

Цель анализа — посмотреть структуру датасета, распределение классов,
качество разметки и понять, подходит ли HateXplain для нашего проекта
по классификации голосовых сообщений.

Источник:
https://github.com/hate-alert/HateXplain

In [1]:
import json
from pprint import pprint

from google.colab import files
uploaded = files.upload()

Saving dataset.json to dataset.json


In [ ]:
with open("dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Тип объекта:", type(data))
print("Количество записей:", len(data))

first_id = next(iter(data))
first_record = data[first_id]

print("\nID первой записи:")
print(first_id)

print("\nПоля записи:")
print(first_record.keys())

print("\nПервая запись целиком:")
pprint(first_record)

Тип объекта: <class 'dict'>
Количество записей: 20148

ID первой записи:
1179055004553900032_twitter

Поля записи:
dict_keys(['post_id', 'annotators', 'rationales', 'post_tokens'])

Первая запись целиком:
{'annotators': [{'annotator_id': 1, 'label': 'normal', 'target': ['None']},
                {'annotator_id': 2, 'label': 'normal', 'target': ['None']},
                {'annotator_id': 3, 'label': 'normal', 'target': ['None']}],
 'post_id': '1179055004553900032_twitter',
 'post_tokens': ['i',
                 'dont',
                 'think',
                 'im',
                 'getting',
                 'my',
                 'baby',
                 'them',
                 'white',
                 '9',
                 'he',
                 'has',
                 'two',
                 'white',
                 'j',
                 'and',
                 'nikes',
                 'not',
                 'even',
                 'touched'],
 'rationales': []}


In [ ]:
import pandas as pd
from collections import Counter

rows = []

for post_id, record in data.items():
    labels = [ann["label"] for ann in record["annotators"]]

    label_counts = Counter(labels)

    majority_label, majority_count = label_counts.most_common(1)[0]

    final_label = majority_label if majority_count >= 2 else None

    text = " ".join(record["post_tokens"])

    rows.append({
        "post_id": post_id,
        "text": text,
        "label_1": labels[0],
        "label_2": labels[1],
        "label_3": labels[2],
        "final_label": final_label,
        "num_tokens": len(record["post_tokens"])
    })

df = pd.DataFrame(rows)

print("Размер таблицы:", df.shape)
display(df.head())

Размер таблицы: (20148, 7)


,post_id,text,label_1,label_2,label_3,final_label,num_tokens
0,1179055004553900032_twitter,i dont think im getting my baby them white 9 h...,normal,normal,normal,normal,20
1,1179063826874032128_twitter,we cannot continue calling ourselves feminists...,normal,normal,normal,normal,42
2,1178793830532956161_twitter,nawt yall niggers ignoring me,normal,normal,hatespeech,normal,5
3,1179088797964763136_twitter,<user> i am bit confused coz chinese ppl can n...,hatespeech,offensive,hatespeech,hatespeech,26
4,1179085312976445440_twitter,this bitch in whataburger eating a burger with...,hatespeech,hatespeech,offensive,hatespeech,20


In [ ]:
print("Распределение итоговых меток:")
print(df["final_label"].value_counts(dropna=False))

print("\nВ процентах:")
print(
    df["final_label"]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
)

Распределение итоговых меток:
final_label
normal        7814
hatespeech    5935
offensive     5480
None           919
Name: count, dtype: int64

В процентах:
final_label
normal        38.78
hatespeech    29.46
offensive     27.20
None           4.56
Name: proportion, dtype: float64


In [ ]:
print("Количество записей без большинства:",
      df["final_label"].isna().sum())

Количество записей без большинства: 919


In [ ]:
df_clean = df.dropna(subset=["final_label"]).copy()

print("Количество записей после удаления спорных:", len(df_clean))

Количество записей после удаления спорных: 19229


In [ ]:
pd.set_option("display.max_colwidth", 300)

for label in ["normal", "offensive", "hatespeech"]:
    print("\n" + "=" * 80)
    print(f"КЛАСС: {label.upper()}")
    print("=" * 80)

    sample = (
        df_clean[df_clean["final_label"] == label]
        .sample(10, random_state=42)
        [["text"]]
    )

    for i, text in enumerate(sample["text"], 1):
        print(f"\n{i}. {text}")


КЛАСС: NORMAL

1. <user> as harsh saying the best way to honour the murdered mp jo cox would be to get brexit done comparing black people to pickaninny muslim women to letterboxes no i do not believe it harsh at all

2. the promise of a real place to go after high school for fat and queer kids could literally keep them alive

3. white gay men mad they can not say dyke bc they do not have faggot to themselves lmao

4. headline says sex robots will replace men only for democrat women decent women still prefer their husbands can democrats get dumber is it possible

5. harden is a joke that nigga gassed all you niggas with that step back three just like westbrook gassed you niggas with triple doubles

6. that charlottesville violence is aweful for years the rightys refrained from violence while the islamic extremists and registered democrats perpetrated it more times than a guy can count on two hands it does not help em that anytime folks tries to assemble the leftys show up to egg people

In [ ]:
def get_agreement(row):
    labels = [row["label_1"], row["label_2"], row["label_3"]]
    counts = Counter(labels)
    return counts.most_common(1)[0][1]

df["agreement"] = df.apply(get_agreement, axis=1)

print("Согласие аннотаторов:")
print(df["agreement"].value_counts().sort_index())

print("\nВ процентах:")
print(
    df["agreement"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

Согласие аннотаторов:
agreement
1     919
2    9384
3    9845
Name: count, dtype: int64

В процентах:
agreement
1     4.56
2    46.58
3    48.86
Name: proportion, dtype: float64


In [ ]:
df_unanimous = df[df["agreement"] == 3].copy()

print("Полностью согласованных записей:", len(df_unanimous))

print("\nРаспределение классов:")
print(df_unanimous["final_label"].value_counts())

print("\nРаспределение классов в процентах:")
print(
    df_unanimous["final_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Полностью согласованных записей: 9845

Распределение классов:
final_label
normal        5124
hatespeech    2960
offensive     1761
Name: count, dtype: int64

Распределение классов в процентах:
final_label
normal        52.05
hatespeech    30.07
offensive     17.89
Name: proportion, dtype: float64


In [ ]:
print(df_clean["num_tokens"].describe())

print("\nКвантили:")
print(
    df_clean["num_tokens"]
    .quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

print("\nКоличество коротких сообщений:")
for threshold in [10, 20, 30, 50, 100]:
    count = (df_clean["num_tokens"] <= threshold).sum()
    pct = count / len(df_clean) * 100
    print(f"<= {threshold:3d} токенов: {count:5d} ({pct:.2f}%)")

count    19229.000000
mean        23.432160
std         13.786932
min          2.000000
25%         12.000000
50%         20.000000
75%         34.000000
max        165.000000
Name: num_tokens, dtype: float64

Квантили:
0.25    12.0
0.50    20.0
0.75    34.0
0.90    45.0
0.95    49.0
0.99    52.0
Name: num_tokens, dtype: float64

Количество коротких сообщений:
<=  10 токенов:  3916 (20.37%)
<=  20 токенов:  9615 (50.00%)
<=  30 токенов: 13507 (70.24%)
<=  50 токенов: 18789 (97.71%)
<= 100 токенов: 19228 (99.99%)


In [ ]:
length_by_class = (
    df_clean
    .groupby("final_label")["num_tokens"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

display(length_by_class)

,count,mean,median,min,max
final_label,,,,,
hatespeech,5935,24.57,22.0,2,165
normal,7814,23.32,20.0,2,70
offensive,5480,22.36,19.0,2,76


In [ ]:
display(
    df_clean
    .sort_values("num_tokens", ascending=False)
    [["final_label", "num_tokens", "text"]]
    .head(10)
)

,final_label,num_tokens,text
7266,hatespeech,165,i would have taken them down myself fucking queers have no business with the military break a finger nail high heel breaks run in the stocking the ass is not a vigina man sucking another man cock gulping his semen is 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮 🤮...
5775,hatespeech,77,a reply that chump dump change obama nigger on the bongos exploded buildings murdering people as leader of a black panther group bruce kenneway bishop with christ and mother mary 🍉 🍍 🍎 🥑 🥜 🥒 🍒 🌽 🥕 🍐 🥔 🍏 🍇 🍔 🍟 🍕 🍕 🌭 🌮 🥞 🧀 🍗 🥙 🌯 <angel> 🏞 ️ 🗽 ⛪ 🕌 🎢 🚲 🚜 ⛽ 🚀 🛫 🛩 ️ 🚁 ⏲️ ☀️ 🌝 🌞 ⛱️ 🌟 🌤 ️
8731,offensive,76,boo sorry did i scare you wassup gurl 😉 😉 😊 its cocktober 😈 🌚 🍂 🍃 🍁 and if you 👈 🏽 are getting this 👇 🏽 😘 it means ur a halloween 👻 🎃 hoe 😏 😩 👅 💦 every year in cocktober the jack o slut 🎃 comes to life 🙀 😻 🙌 🏽 👏 👏 🙌 🏽 coming to harvest 🍁 🍂 🍃 his hoes for thot -o- ween 😏 😏 💥
1625,normal,70,young imams of mosques will be prepared under the direct control of white house officials such conspiracy against islam is introduced in saudia recently in pakistan it is long been implemented in private school إ ِ ن َ ّ ا ل ِ ل ّ ه ِ و َ إ ِ ن َ ّ ـا إ ِ ل َ ي ْ ه ِ ر َ اج ِ عون watch the video
9133,normal,67,<user> antifa are true patriots these are the brave men and women who show up to confront white supremacists and neo nazis while white supremacists are killing people the right wing media gets more upset because antifa blocked a doorway 🤦 ‍ ♂️ 🤦 ‍ ♂️ 🤦 ‍ ♂️ 🤦 ‍ ♂️ 🤦 ‍ ♂️ 🤦 ‍ ♂️ 🤦 ‍ ♂️ 🤦 ‍ ♂️ 🤦 ‍ ♂️
1176,normal,63,boo 👻 👺 👻 sorry did i scare you 🙃 🙃 it cocktober 🍆 😈 🌚 u kno what that means slut -o- ween send this to <number> of ur sluttiest ghost 👻 bitches 🙎 if u get <number> back ur a slutty witch 🔮 bitch but if u get <number> back ur the spookiest slut on the block 🌚 😈 🔮 💯
17507,hatespeech,60,retard usa needs just let these fucks kill them selves they been fighting for 2 0 0 0 years assholes will never get along lmfao 😂 😂 😃 it so silly pretend like your going save these dumbass sandniggers let just do what we came for and fuck iran up quit bullshit 💀 💀 💀 💀 💀 💀 💀 💀
7860,hatespeech,60,hail victory white lives matter make america white again stop white genocide this is proof what future arayn race has to face nsm 88 kk nazi american partys wail you all do nothing this is what each day draws near it does exsit black supremacy is africa new sloggen and the nigger jews here in am...
8711,normal,58,<user> <user> <user> <user> <user> <user> <user> <user> <user> the hardest thing for any refugee parents is this question from their children why do not we have a homeland this is the most painful question for iraqi refugees whose children grew up in the host countries they ask for stability wil...
12547,normal,57,muslim girls express how horny they are by saying they can ’ t wait to have kids 🤷 🏻 ‍ ♀ ️ 🤷 🏻 ‍ ♀ ️ 🤷 🏻 ‍ ♀ ️ 🤷 🏻 ‍ ♀ ️ 🤷 🏻 ‍ ♀ ️ 🤷 🏻 ‍ ♀ ️ 🤷 🏻 ‍ ♀ ️ 🤷 🏻 ‍ ♀ ️


In [ ]:
import re

# Источник сообщения
df_clean["source"] = df_clean["post_id"].apply(
    lambda x: "twitter" if x.endswith("_twitter") else "gab"
)

# Наличие специальных токенов
df_clean["has_user"] = df_clean["text"].str.contains("<user>", regex=False)
df_clean["has_number"] = df_clean["text"].str.contains("<number>", regex=False)
df_clean["has_percent"] = df_clean["text"].str.contains("<percent>", regex=False)

# Любой символ вне ASCII
df_clean["has_non_ascii"] = df_clean["text"].apply(
    lambda x: any(ord(ch) > 127 for ch in x)
)

print("Источники:")
print(df_clean["source"].value_counts())

print("\nСпециальные элементы:")
for col in ["has_user", "has_number", "has_percent", "has_non_ascii"]:
    count = df_clean[col].sum()
    pct = count / len(df_clean) * 100
    print(f"{col:15s}: {count:5d} ({pct:.2f}%)")

print("\nОчень короткие записи:")
for threshold in [2, 3, 5]:
    count = (df_clean["num_tokens"] <= threshold).sum()
    pct = count / len(df_clean) * 100
    print(f"<= {threshold} токенов: {count} ({pct:.2f}%)")

Источники:
source
gab        10451
twitter     8778
Name: count, dtype: int64

Специальные элементы:
has_user       :  4809 (25.01%)
has_number     :  1927 (10.02%)
has_percent    :   116 (0.60%)
has_non_ascii  :  2267 (11.79%)

Очень короткие записи:
<= 2 токенов: 4 (0.02%)
<= 3 токенов: 183 (0.95%)
<= 5 токенов: 995 (5.17%)


In [ ]:
quality_by_class = (
    df_clean
    .groupby("final_label")
    .agg(
        count=("post_id", "count"),
        user_placeholder=("has_user", "mean"),
        number_placeholder=("has_number", "mean"),
        percent_placeholder=("has_percent", "mean"),
        non_ascii=("has_non_ascii", "mean")
    )
)

for col in [
    "user_placeholder",
    "number_placeholder",
    "percent_placeholder",
    "non_ascii"
]:
    quality_by_class[col] *= 100

display(quality_by_class.round(2))

,count,user_placeholder,number_placeholder,percent_placeholder,non_ascii
final_label,,,,,
hatespeech,5935,10.87,9.71,0.78,6.22
normal,7814,37.30,10.69,0.72,14.91
offensive,5480,22.79,9.42,0.26,13.38


In [ ]:
print("Полных дубликатов текста:",
      df_clean["text"].duplicated().sum())

print("Уникальных текстов:",
      df_clean["text"].nunique())

duplicates = (
    df_clean[df_clean["text"].duplicated(keep=False)]
    .sort_values("text")
    [["final_label", "text"]]
)

display(duplicates.head(20))

Полных дубликатов текста: 37
Уникальных текстов: 19192


,final_label,text
13860,normal,<user> <number> bank islam
13853,normal,<user> <number> bank islam
13851,normal,<user> <number> bank islam
12304,offensive,<user> <user> are you retarded
10987,offensive,<user> <user> are you retarded
12023,offensive,<user> are you a fucking retard
11994,offensive,<user> are you a fucking retard
4896,offensive,<user> ching chong
9932,hatespeech,<user> ching chong
9311,normal,<user> i fucking hate you


In [ ]:
label_conflicts = (
    df_clean
    .groupby("text")["final_label"]
    .nunique()
)

conflicting_texts = label_conflicts[label_conflicts > 1]

print(
    "Уникальных одинаковых текстов с разными метками:",
    len(conflicting_texts)
)

Уникальных одинаковых текстов с разными метками: 6


In [ ]:
conflict_examples = (
    df_clean[df_clean["text"].isin(conflicting_texts.index)]
    .sort_values("text")
    [["text", "final_label", "label_1", "label_2", "label_3"]]
)

display(conflict_examples)

,text,final_label,label_1,label_2,label_3
4896,<user> ching chong,offensive,offensive,offensive,normal
9932,<user> ching chong,hatespeech,hatespeech,hatespeech,normal
3573,<user> i fucking hate you,normal,normal,normal,normal
13180,<user> i fucking hate you,offensive,hatespeech,offensive,offensive
9311,<user> i fucking hate you,normal,normal,normal,hatespeech
14091,<user> i hate you,hatespeech,hatespeech,hatespeech,offensive
12504,<user> i hate you,normal,normal,normal,hatespeech
12405,<user> i hate you,normal,normal,normal,hatespeech
12147,<user> i hate you,hatespeech,hatespeech,hatespeech,offensive
10970,<user> i hate you,normal,hatespeech,normal,normal


In [ ]:
import re

def simple_tts_normalize(text):
    text = text.lower()

    # Убираем служебные placeholders пользователей
    text = text.replace("<user>", " ")

    # Нормализуем placeholders чисел
    text = text.replace("<number>", " number ")
    text = text.replace("<percent>", " percent ")

    # Для первого анализа убираем emoji и прочие non-ASCII символы
    text = "".join(ch for ch in text if ch.isascii())

    # Убираем лишние пробелы
    text = re.sub(r"\s+", " ", text).strip()

    return text


df_clean["tts_text_simple"] = df_clean["text"].apply(simple_tts_normalize)

print("Уникальных исходных текстов:",
      df_clean["text"].nunique())

print("Уникальных после TTS-нормализации:",
      df_clean["tts_text_simple"].nunique())

print("Дубликатов после нормализации:",
      df_clean["tts_text_simple"].duplicated().sum())

Уникальных исходных текстов: 19192
Уникальных после TTS-нормализации: 19163
Дубликатов после нормализации: 66


In [ ]:
normalized_conflicts = (
    df_clean
    .groupby("tts_text_simple")["final_label"]
    .nunique()
)

normalized_conflicts = normalized_conflicts[
    normalized_conflicts > 1
]

print(
    "Нормализованных текстов с разными классами:",
    len(normalized_conflicts)
)

Нормализованных текстов с разными классами: 10


In [ ]:
display(
    df_clean[
        df_clean["tts_text_simple"].isin(normalized_conflicts.index)
    ][
        ["tts_text_simple", "text", "final_label"]
    ]
    .sort_values("tts_text_simple")
    .head(50)
)

,tts_text_simple,text,final_label
627,biggest coward chinaman satanist lee hsien loong criminals had audible talks past few years it started in number he got his criminals to loiter at my 1 9 8 0 address they gossiped false information around number number and number marsiling lane singapore,<user> biggest coward chinaman satanist lee hsien loong criminals had audible talks past few years it started in <number> he got his criminals to loiter at my 1 9 8 0 address they gossiped false information around <number> <number> and <number> marsiling lane singapore,normal
8281,biggest coward chinaman satanist lee hsien loong criminals had audible talks past few years it started in number he got his criminals to loiter at my 1 9 8 0 address they gossiped false information around number number and number marsiling lane singapore,<user> <user> biggest coward chinaman satanist lee hsien loong criminals had audible talks past few years it started in <number> he got his criminals to loiter at my 1 9 8 0 address they gossiped false information around <number> <number> and <number> marsiling lane singapore,hatespeech
9859,bitch i hate you,<user> bitch i hate you 😂 😂 😂 😂,normal
3756,bitch i hate you,<user> bitch i hate you,offensive
9932,ching chong,<user> ching chong,hatespeech
4896,ching chong,<user> ching chong,offensive
4708,he dindu nuffin,<user> he dindu nuffin,hatespeech
9498,he dindu nuffin,<user> <user> <user> he dindu nuffin,normal
803,he dindu nuffin,<user> <user> he dindu nuffin,normal
13180,i fucking hate you,<user> i fucking hate you,offensive


In [ ]:
source_class = pd.crosstab(
    df_clean["source"],
    df_clean["final_label"],
    margins=True
)

display(source_class)

final_label,hatespeech,normal,offensive,All
source,,,,
gab,5236,2058,3157,10451
twitter,699,5756,2323,8778
All,5935,7814,5480,19229


In [ ]:
source_class_pct = pd.crosstab(
    df_clean["source"],
    df_clean["final_label"],
    normalize="index"
).mul(100).round(2)

display(source_class_pct)

final_label,hatespeech,normal,offensive
source,,,
gab,50.10,19.69,30.21
twitter,7.96,65.57,26.46


## Вывод

HateXplain содержит 20 148 сообщений с тремя классами: hatespeech, offensive и normal.

После удаления записей без большинства среди трёх аннотаторов остаётся 19 229 сообщений.

Большая часть сообщений короткая: медианная длина составляет 20 токенов, а около 98% сообщений содержат не более 50 токенов. Поэтому датасет подходит для дальнейшего преобразования текста в аудио с помощью TTS.

Перед использованием потребуется очистить служебные элементы вроде <user>, <number> и эмодзи, а также убрать небольшое количество конфликтующих дубликатов.

Также был обнаружен заметный перекос классов между источниками Gab и Twitter. При подготовке итоговой выборки это нужно будет учесть и сбалансировать данные.